# Public SEC Covenant Monitoring RAG

A project-specific clone of `02-rag.ipynb`. It retrieves public SEC document records for analyst review of covenant terms, reported leverage, compliance events, amendments, and waivers. The pipeline keeps the course structure: `search`, `build_prompt`, `llm`, and `rag`.

In [1]:
import os
from functools import partial
from pathlib import Path

from credit_monitoring.ingestion.chunking.evidence import chunk_evidence_documents
from credit_monitoring.ingestion.loaders.sec import build_company_catalog, create_documents
from credit_monitoring.observability.logging import configure_logging
from credit_monitoring.retrieval.lexical.sec import build_index

# Timestamped document/section progress and timings in cell output.
configure_logging()

# Explicit filesystem configuration; reusable module imports have no side effects.
data_dir = Path("data")
if not data_dir.exists():
    data_dir = Path("../data")
html_dir = data_dir / "sec_documents" / "html"
markdown_dir = data_dir / "sec_documents" / "markdown"
source_file = data_dir / "public_sec_documents.csv"


In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()


## 1. Download, clean, convert, and load the selected documents

The loader downloads each HTML filing listed in the selected CSV, saves the original HTML, removes clear page chrome with Beautiful Soup, and converts the cleaned HTML to Markdown. It then selects covenant-related evidence lines with neighboring context and indexes the resulting Markdown chunks with the catalog metadata.

SEC requests need an identifying `SEC_USER_AGENT` environment variable containing the requester's name or organization and a real contact email. Cached HTML and Markdown files can be reused. The first run after adding HTML cleanup reconverts cached HTML so older Markdown does not bypass cleanup.


In [3]:
# Downloads use your configured identity; cached filings need no network request.
sec_user_agent = os.environ.get("SEC_USER_AGENT", "")


In [4]:
from markitdown import MarkItDown

converter = MarkItDown(enable_plugins=False)


2026-10-07 12:44:13.930177916 [W:onnxruntime:Default, device_discovery.cc:146 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


## 2. Build the Index

Index covenant-related evidence excerpts with issuer and catalog metadata. Each result retains the filing citation and Markdown path for verification.


In [5]:
filing_documents = create_documents(
    source_file,
    html_dir=html_dir,
    markdown_dir=markdown_dir,
    user_agent=sec_user_agent,
    converter=converter,
)
documents = chunk_evidence_documents(filing_documents, size=3500, step=500)
index = build_index(documents)
print(f"Loaded {len(filing_documents)} filings and {len(documents)} evidence chunks.")
print(f"Original HTML: {html_dir} | Markdown: {markdown_dir}")


2026-10-07 12:44:17,938 INFO credit_monitoring.ingestion.converters.sec_markdown: Markdown cache hit document_id=DOC_FMC_AMD3_2025
2026-10-07 12:44:17,944 INFO credit_monitoring.ingestion.converters.sec_markdown: Markdown cache hit document_id=DOC_FMC_Q1_2025
2026-10-07 12:44:18,000 INFO credit_monitoring.ingestion.converters.sec_markdown: Markdown cache hit document_id=DOC_FMC_Q3_2025
2026-10-07 12:44:18,171 INFO credit_monitoring.ingestion.converters.sec_markdown: Markdown cache hit document_id=DOC_HAIN_Q1_2025
2026-10-07 12:44:18,216 INFO credit_monitoring.ingestion.converters.sec_markdown: Markdown cache hit document_id=DOC_HAIN_AMD3_2025
2026-10-07 12:44:18,224 INFO credit_monitoring.ingestion.converters.sec_markdown: Markdown cache hit document_id=DOC_HAIN_10K_2025
2026-10-07 12:44:18,340 INFO credit_monitoring.ingestion.converters.sec_markdown: Markdown cache hit document_id=DOC_HAIN_Q3_2025
2026-10-07 12:44:18,446 INFO credit_monitoring.ingestion.converters.sec_markdown: Markdo

Loaded 29 filings and 53 evidence chunks.
Original HTML: ../data/sec_documents/html | Markdown: ../data/sec_documents/markdown


In [6]:
documents[0] if documents else "No evidence excerpts were selected."


{'start': 0,
 'content': '(a)Section 1.01 of the Credit Agreement is hereby amended by amending and restating the definition of “Covenant Relief Period Termination Date” in its entirety as follows:\n“Covenant Relief Period Termination Date” means the earlier of (a) December 31, 2027 and (b) the date on which the Administrative Agent receives a Covenant Relief Period Termination Notice from the U.S. Borrower; provided that, with respect to clause (b) hereof, no Event of Default or Default shall have occurred and be continuing.\nMaximum Leverage Ratio. The U.S. Borrower shall maintain as of the last day of each Fiscal Quarter a Leverage Ratio of not more than 3.75:1.00; provided, however, that notwithstanding the foregoing, during the Covenant Relief Period, the U.S. Borrower shall maintain as of the last day of each Fiscal Quarter a Leverage Ratio of not more than the applicable level set forth below adjacent to such Fiscal Quarter:\n|  |  |  |  |  |  |\n| Fiscal Quarter | | | Maximum L

## 3. Import and bind the RAG functions

Reusable functions live in `src/credit_monitoring`. Bind the explicit client and index here; prompts are imported from dedicated modules. Narrative responses report source-stated facts without calculating ratios or headroom.


In [7]:
from credit_monitoring.retrieval.lexical.sec import search as search_records

search = partial(search_records, index=index, document_count=len(documents))


In [8]:
from credit_monitoring.agents.prompts.assembly import build_prompt
from credit_monitoring.agents.prompts.narrative import NARRATIVE_INSTRUCTIONS as instructions


In [9]:
from credit_monitoring.application.rag_service import llm as generate_narrative

llm = partial(generate_narrative, client=openai_client)


In [10]:
from credit_monitoring.application.rag_service import rag as run_rag

rag = partial(run_rag, client=openai_client, index=index, document_count=len(documents))


In [11]:
search_results = search( """
            For FMC Corporation (FMC), summarize the reported net leverage trend and covenant status in the available SEC records. 
            Cite each record and explain what evidence is still needed to assess current covenant headroom.
        """)

print(search_results)

[{'start': 0, 'content': '(a)Section 1.01 of the Credit Agreement is hereby amended by amending and restating the definition of “Covenant Relief Period Termination Date” in its entirety as follows:\n“Covenant Relief Period Termination Date” means the earlier of (a) December 31, 2027 and (b) the date on which the Administrative Agent receives a Covenant Relief Period Termination Notice from the U.S. Borrower; provided that, with respect to clause (b) hereof, no Event of Default or Default shall have occurred and be continuing.\nMaximum Leverage Ratio. The U.S. Borrower shall maintain as of the last day of each Fiscal Quarter a Leverage Ratio of not more than 3.75:1.00; provided, however, that notwithstanding the foregoing, during the Covenant Relief Period, the U.S. Borrower shall maintain as of the last day of each Fiscal Quarter a Leverage Ratio of not more than the applicable level set forth below adjacent to such Fiscal Quarter:\n|  |  |  |  |  |  |\n| Fiscal Quarter | | | Maximum L

## 4. Try It Out

Both prompts search the Markdown evidence chunks and include up to two relevant chunks per filing in the LLM context.


### Prompt 1: FMC borrower review


In [12]:
print(
    rag(
        """
            For FMC Corporation (FMC), summarize the reported net leverage trend and covenant status in the available SEC records. 
            Cite each record and explain what evidence is still needed to assess current covenant headroom.
        """
    )
)


### Covenant Monitoring Notes: FMC Corporation

#### Document Excerpts:
1. **Credit Agreement Amendment No. 3**
   - **Citation**: FMC Corporation, Amendment No. 3 (2025-02-03), [Link](https://www.sec.gov/Archives/edgar/data/37785/000003778525000013/amendmentno3tofifthamended.htm)
   - **Leverage Ratios**:
     - Maximum Leverage Ratio of 4.00:1.00 for June 30, 2023.
     - Maximum Leverage Ratio of 3.75:1.00 as of December 31, 2027.
   - **Interest Coverage Ratios**:
     - Minimum Interest Coverage Ratio of 2.50:1.00 for December 31, 2023.
     - Minimum Interest Coverage Ratio of 3.50:1.00 thereafter.
   - **Compliance Statement**: No Defaults or Events of Default at the Amendment's effective date.
   - **Notes**: Introduces a Covenant Relief Period (documented relief from ratio requirements).

2. **10-Q for Period Ending September 30, 2025**
   - **Citation**: FMC Corporation, 10-Q (2025-09-30), [Link](https://www.sec.gov/Archives/edgar/data/37785/000003778525000127/fmc-20250930.ht

### Prompt 2: Full-corpus comparison table


In [13]:
print(
    rag(
        """
            Across all indexed SEC filings, compare each issuer's latest disclosed leverage covenant result or status with 
            available prior periods. Present the findings in a concise table for quick analyst review, with columns for issuer, 
            latest period, leverage result, covenant threshold, compliance or waiver status, comparison with prior period, 
            and source citation. Include applicable thresholds and relevant amendments or waivers where the filings support them. 
            Mark unavailable information as "Not disclosed in retrieved sources," and do not infer missing values or status.
        """
    )
)

| Issuer                       | Latest Period     | Leverage Result      | Covenant Threshold   | Compliance/Waiver Status | Comparison with Prior Period   | Source Citation                                                                                     |
|------------------------------|-------------------|-----------------------|----------------------|---------------------------|--------------------------------|-----------------------------------------------------------------------------------------------------|
| Stoneridge, Inc.             | Dec 31, 2024      | Less than 3.50       | 3.50                 | Event of Default          | Not disclosed in retrieved sources | Stoneridge, Inc., Amendment No. 1 + Waiver (2025-02-26), DOC_STONE_AMD1_2025                     |
| The Hain Celestial Group, Inc.| Q1 2025           | 5.00                  | 5.00 (until Dec 31, 2024) | Compliance               | Not disclosed in retrieved sources | The Hain Celestial Group, Inc., 10-Q (2025-

## 5. Process complete documents and reuse stored facts

Document processing is independent of question-based retrieval. The synchronous service
reads every Markdown section, extracts all relevant facts, and stores successful results
in SQLite. Repeated calls reuse the document/version cache. Automatic verification is
temporarily disabled: result wrappers use schema version 2.1 and `validation: null`.


In [14]:
from credit_monitoring.application.document_processing_service import DocumentProcessingService

processing_service = DocumentProcessingService(
    client=openai_client,
    source_file=source_file,
    html_dir=html_dir,
    markdown_dir=markdown_dir,
    database_path=data_dir / "processed" / "covenants.sqlite3",
    user_agent=sec_user_agent,
    converter=converter,
)


In [15]:
# Discover registered documents, including missing or stale extractions; no model call.
[
    (document.document_id, document.status)
    for document in processing_service.list_document_extractions(ticker="FMC")
]


[('DOC_FMC_AMD3_2025', 'complete'),
 ('DOC_FMC_Q1_2025', 'complete'),
 ('DOC_FMC_Q3_2025', 'complete')]

In [16]:
# One complete filing, including all leverage and interest-coverage schedule rows.
# A cache hit makes no model calls. Explicit force=True requests fresh processing.
fmc_document = processing_service.process_document("DOC_FMC_AMD3_2025")
fmc_extraction = fmc_document.result
print("Cache reused:", fmc_document.cache_hit)
print("Sections completed:", fmc_document.completed_sections, "/", fmc_document.total_sections)
print(fmc_extraction.model_dump_json(indent=2))
print("Verification:", "not run" if fmc_extraction.validation is None else fmc_extraction.validation)


2026-10-07 12:45:55,669 INFO credit_monitoring.application.document_processing_service: Process document started document_id=DOC_FMC_AMD3_2025 force=False
2026-10-07 12:45:55,673 INFO credit_monitoring.application.document_processing_service: Load Markdown started document_id=DOC_FMC_AMD3_2025
2026-10-07 12:45:55,674 INFO credit_monitoring.ingestion.converters.sec_markdown: Markdown cache hit document_id=DOC_FMC_AMD3_2025
2026-10-07 12:45:55,674 INFO credit_monitoring.application.document_processing_service: Load Markdown complete document_id=DOC_FMC_AMD3_2025 elapsed_s=0.00
2026-10-07 12:45:55,675 INFO credit_monitoring.application.document_processing_service: Check extraction cache started document_id=DOC_FMC_AMD3_2025
2026-10-07 12:45:55,680 INFO credit_monitoring.application.document_processing_service: Check extraction cache complete document_id=DOC_FMC_AMD3_2025 elapsed_s=0.00
2026-10-07 12:45:55,681 INFO credit_monitoring.application.document_processing_service: Extraction cache

Cache reused: True
Sections completed: 1 / 1
{
  "schema_version": "2.1",
  "prompt_version": "sec-covenant-extraction-v2.1",
  "extraction": {
    "issuer": "FMC Corporation",
    "ticker": "FMC",
    "borrower": "FMC Corporation",
    "guarantors": [],
    "agreements": [
      {
        "agreement_name": "Fifth Amended and Restated Credit Agreement",
        "amendment_name": "Amendment No. 3",
        "amendment_number": "3",
        "execution_date": "2025-02-03",
        "stated_effective_date": "2025-02-03",
        "parties": [
          "FMC CORPORATION",
          "certain of the Company’s subsidiaries from time to time party thereto as borrowers",
          "CITIBANK, N.A. as Administrative Agent",
          "each lender and issuing bank from time to time party thereto"
        ],
        "referenced_documents": [
          "Amendment No. 1, dated as of June 30, 2023",
          "Amendment No. 2, dated as of November 7, 2023"
        ],
        "evidence": {
          "evide

In [19]:
documents = processing_service.process_all_documents(force=False)

2026-10-07 12:46:34,544 INFO credit_monitoring.application.document_processing_service: Process all documents started ticker=None force=False
2026-10-07 12:46:34,554 INFO credit_monitoring.application.document_processing_service: Load and validate catalog started 
2026-10-07 12:46:34,561 INFO credit_monitoring.application.document_processing_service: Load and validate catalog complete  elapsed_s=0.01
2026-10-07 12:46:34,570 INFO credit_monitoring.application.document_processing_service: Batch ready total_documents=29 ticker=None force=False
2026-10-07 12:46:34,570 INFO credit_monitoring.application.document_processing_service: Batch document started document=1/29 document_id=DOC_FMC_AMD3_2025 ticker=FMC
2026-10-07 12:46:34,578 INFO credit_monitoring.application.document_processing_service: Process document started document_id=DOC_FMC_AMD3_2025 force=False
2026-10-07 12:46:34,583 INFO credit_monitoring.application.document_processing_service: Load Markdown started document_id=DOC_FMC_AM

In [20]:
print(documents[-1].model_dump_json(indent=2))

{
  "document_id": "DOC_TILE_2025_10K",
  "citation": "Tile Shop Holdings, Inc., 10-K (2025-12-31), DOC_TILE_2025_10K — https://www.sec.gov/Archives/edgar/data/1552800/000155280026000006/c800-20251231x10k.htm",
  "ticker": "TTSH",
  "status": "complete",
  "cache_key": "3e187ecd1571103e2b8dac8e9080c795ec54f4ec1bce17c9930e38a1851ff5ad",
  "cache_hit": true,
  "completed_sections": 7,
  "total_sections": 7,
  "result": {
    "schema_version": "2.1",
    "prompt_version": "sec-covenant-extraction-v2.1",
    "extraction": {
      "issuer": "Tile Shop Holdings, Inc.",
      "ticker": "TTSH",
      "borrower": null,
      "guarantors": [],
      "agreements": [
        {
          "agreement_name": "10-K",
          "amendment_name": null,
          "amendment_number": null,
          "execution_date": null,
          "stated_effective_date": "2025-12-31",
          "parties": [
            "Tile Shop Holdings, Inc."
          ],
          "referenced_documents": [],
          "evidence": {


### Process selected catalog documents

The application can call the same service as soon as a document arrives or requires
reprocessing. No queue or worker is required. This helper processes registered documents
sequentially; invoke it explicitly to avoid starting full-corpus model calls accidentally.


In [21]:
def process_catalog(ticker=None, force=False):
    return processing_service.process_all_documents(ticker=ticker, force=force)

# Optional: process all registered FMC filings, reusing completed results.
# fmc_documents = process_catalog(ticker="FMC")
# Optional: process the complete registered catalog.
# all_documents = process_catalog()


## 6. Credit assessment agent: document research

The OpenAI Agents SDK runs the tool loop. The agent chooses among listing documents,
reading stored extractions, searching source
passages, and processing a document when needed. Each tool result returns to the model;
the model chooses another call or a cited answer. At most six tool rounds execute.
The public trace shows actual calls and outcomes, never private model reasoning.

This question requests complete stored schedules and supporting filing passages. Actual
tool selection remains model-driven; the tests exercise both paths with mocked responses.
The earlier direct `rag()` function remains available independently.


In [22]:
from credit_monitoring.agents.credit_assessment import CreditAssessmentAgent

credit_assessment_agent = CreditAssessmentAgent(client=openai_client, service=processing_service)


In [23]:
agent_answer = await credit_assessment_agent.ask_async(
    "Read the complete leverage and interest-coverage schedules in FMC Amendment No. 3. "
    "Include every explicitly named fiscal quarter, then search the FMC filings for "
    "supporting passages and explicitly reported ratios. Cite the sources. "
    "Do not determine which amendment governs or calculate headroom.",
    ticker="FMC",
)
print(agent_answer.answer)
print("Completed:", agent_answer.complete, "Error:", agent_answer.error)


Here are the complete leverage and interest coverage schedules from **FMC Corporation's Amendment No. 3** and supporting filings.

### Leverage Ratio Schedule

**Amendment No. 3 Periods and Ratios**
- **Fiscal Quarter Ended June 30, 2023**: Maximum Leverage Ratio = **4.00 to 1.00**
- **Fiscal Quarter Ended September 30, 2023**: Maximum Leverage Ratio = **4.00 to 1.00**
- **Fiscal Quarter Ended December 31, 2023**: Maximum Leverage Ratio = **6.50 to 1.00**
- **Fiscal Quarter Ended March 31, 2024**: Maximum Leverage Ratio = **6.50 to 1.00**
- **Fiscal Quarter Ended June 30, 2024**: Maximum Leverage Ratio = **6.50 to 1.00**
- **Fiscal Quarter Ended September 30, 2024**: Maximum Leverage Ratio = **6.00 to 1.00**
- **Fiscal Quarter Ended December 31, 2024**: Maximum Leverage Ratio = **5.00 to 1.00**
- **Fiscal Quarter Ended March 31, 2025**: Maximum Leverage Ratio = **5.25 to 1.00**
- **Fiscal Quarter Ended June 30, 2025**: Maximum Leverage Ratio = **5.25 to 1.00**
- **Fiscal Quarter Ended 

In [24]:
# Inspect the actual tool execution sequence, including cache and failure outcomes.
for execution in agent_answer.tool_trace:
    print(execution.name, execution.arguments, "ok:", execution.outcome["ok"])

##print(documents[-1].model_dump_json(indent=2))
print([ str(execution.model_dump()) for execution in agent_answer.tool_trace])

# The complete structured outcome is also serializable:
# print(agent_answer.model_dump_json(indent=2))


list_documents {'ticker': 'FMC'} ok: True
get_document_extraction {'document_id': 'DOC_FMC_AMD3_2025'} ok: True
search_evidence {'ticker': 'FMC', 'query': 'Leverage Ratio'} ok: True
["{'call_id': 'call_7I44Y5wqZ2GLwL0f4N3YlEyN', 'name': 'list_documents', 'arguments': {'ticker': 'FMC'}, 'outcome': {'ok': True, 'data': [{'document_id': 'DOC_FMC_AMD3_2025', 'citation': 'FMC Corporation, Amendment No. 3 (2025-02-03), DOC_FMC_AMD3_2025 — https://www.sec.gov/Archives/edgar/data/37785/000003778525000013/amendmentno3tofifthamended.htm', 'ticker': 'FMC', 'status': 'complete', 'cache_key': '31e34268610242ccbcbe8934aefe9fbd2ec1007c8cb3e7230c823f2a8552175e', 'cache_hit': True, 'completed_sections': 1, 'total_sections': 1, 'error': None}, {'document_id': 'DOC_FMC_Q1_2025', 'citation': 'FMC Corporation, 10-Q (2025-03-31), DOC_FMC_Q1_2025 — https://www.sec.gov/Archives/edgar/data/37785/000003778525000065/fmc-20250331.htm', 'ticker': 'FMC', 'status': 'complete', 'cache_key': '6a9e77fac7791b6a74ac043bd

## 7. Credit assessment agent: calculations and predictions

The same agent can call the existing assessment service for an explicitly scoped borrower.
Python resolves configured covenant terms and calculates ratios, compliance and headroom;
the configured model supplies optional predictions. These synthetic borrowers have no
inferred relationship to the real SEC issuers above.

This example uses separate demonstration databases and the default stub, which supplies
no forecast. To request a real baseline probability, generate/train the model with the
scripts in `docs/ml_early_warning.md` and explicitly select its artifact in the next cell.
The Logistic Regression baseline was trained on simulated histories.


In [25]:
from credit_monitoring.application.assessment_service import AssessmentService
from credit_monitoring.config.settings import AssessmentSettings, RiskModelConfig
from credit_monitoring.covenants.structured import SyntheticCovenantReader
from credit_monitoring.financials.repositories import FinancialRepository
from credit_monitoring.ingestion.loaders.financials import load_financials

agent_financial_db = data_dir / "processed" / "agent_financials.sqlite3"
load_financials(data_dir / "synthetic_quarterly_financials.csv", agent_financial_db)

# Keep model selection in application configuration, outside the agent's arguments.
agent_model = RiskModelConfig(type="stub")
# After training the baseline, replace the line above with:
# agent_model = RiskModelConfig(
#     type="logistic_regression", version="v1",
#     artifact_path=data_dir / "models" / "logistic_regression" / "v1",
# )

assessment_service = AssessmentService(
    financial_repository=FinancialRepository(agent_financial_db),
    covenant_reader=SyntheticCovenantReader(data_dir / "synthetic_covenant_terms.csv"),
    settings=AssessmentSettings(
        analytics_database=data_dir / "processed" / "agent_analytics.sqlite3",
        risk_model=agent_model,
    ),
)
credit_assessment_agent = CreditAssessmentAgent(
    client=openai_client,
    service=processing_service,
    assessment_service=assessment_service,
)


In [26]:
assessment_answer = await credit_assessment_agent.ask_async(
    "Read the financials and assess covenant compliance and headroom for 2025-09-30, "
    "with an information cutoff of 2025-09-30. Also request a next-quarter covenant "
    "breach prediction. Cite financial and covenant provenance, distinguish current "
    "compliance from predictions, and explain if the configured model has no forecast.",
    borrower_id="SYN002",
)
print(assessment_answer.answer)
print("Completed:", assessment_answer.complete, "Error:", assessment_answer.error)
for execution in assessment_answer.tool_trace:
    print(execution.name, execution.arguments, "ok:", execution.outcome["ok"])


### Financial Insights for Borrower ID: SYN002 as of 2025-09-30

#### Financial Metrics
- **Total Debt:** 145.0 (in millions)
- **Cash:** 10.0 (in millions)
- **Reported EBITDA:** 26.0 (in millions)

**Source:** Synthetic Quarterly Financials [Row 7] from `/workspaces/capstone-ai-private-credit/data/synthetic_quarterly_financials.csv`.

#### Covenant Assessment
- **Covenant Type:** Maximum Net Leverage
- **Actual Value:** 5.19
- **Threshold Value:** 5.5
- **Headroom:** 0.31
- **Compliance Status:** Compliant

**Evidence Source:** Synthetic Covenant Terms [Row 3] from `/workspaces/capstone-ai-private-credit/data/synthetic_covenant_terms.csv`.

#### Compliance Summary
The assessment indicates that the borrower is compliant with the Maximum Net Leverage covenant as of 2025-09-30, with a current value (5.19) below the threshold (5.5). The headroom of 0.31 provides a buffer.

### Predictions for Future Compliance
I attempted to predict the next-quarter (2025-12-31) covenant compliance, but 